# Banking Customer & Loan Analytics — Data Cleaning

## 1. Introduction

The objective of this phase is to prepare the banking datasets for further analysis by identifying and handling data quality issues while preserving the original information.

## 2. Cleaning Objectives

- Convert columns into appropriate data types.
- Handle missing values based on business logic.
- Identify and correct inconsistent or invalid values, if any.
- Preserve valid outliers.
- Validate the cleaned datasets.
- Save the cleaned datasets separately from the original raw files.

## 3. Data Cleaning Approach

The cleaning process will be performed on copies of the original datasets to ensure that raw data remains unchanged.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
project_path = Path("..")

raw_path = project_path / "Data" / "raw"
cleaned_path = project_path / "data" / "cleaned"

cleaned_path.mkdir(parents=True, exist_ok=True)

In [3]:
print("Raw Data Path:", raw_path)
print("Raw Folder Exists:", raw_path.exists())

print("Cleaned Data Path:", cleaned_path)
print("Cleaned Folder Exists:", cleaned_path.exists())

Raw Data Path: ..\data\raw
Raw Folder Exists: True
Cleaned Data Path: ..\data\cleaned
Cleaned Folder Exists: True


In [5]:
raw_datasets = {
    'Customers': pd.read_csv(raw_path / 'bank-customers.csv'),
    'Accounts': pd.read_csv(raw_path / 'bank-accounts.csv'),
    'Account_Holders': pd.read_csv(raw_path / 'bank-account-holders.csv'),
    'Transactions': pd.read_csv(raw_path / 'bank-transactions.csv'),
    'Loans': pd.read_csv(raw_path / 'bank-loans.csv'),
    'Cards': pd.read_csv(raw_path / 'bank-cards.csv'),
    'Counties': pd.read_csv(raw_path / 'bank-counties.csv')
}

In [7]:
cleaned_datasets = {
    name: df.copy()
    for name, df in raw_datasets.items()
}

In [8]:
for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")

Customers: (1034, 5)
Accounts: (900, 5)
Account_Holders: (1034, 4)
Transactions: (102880, 8)
Loans: (296, 7)
Cards: (232, 4)
Counties: (15, 7)


In [9]:
for name in raw_datasets:

    is_equal = raw_datasets[name].equals(
        cleaned_datasets[name]
    )

    print(f"{name}: {is_equal}")

Customers: True
Accounts: True
Account_Holders: True
Transactions: True
Loans: True
Cards: True
Counties: True


## Date Columns Data Type Conversion

In [10]:
date_columns = {
    'Accounts': ['opened_date'],
    'Transactions': ['txn_date'],
    'Loans': ['loan_date'],
    'Cards': ['issued_date']
}

for name, columns in date_columns.items():

    print(f"\n{name}")

    for col in columns:
        print(col, ":", cleaned_datasets[name][col].dtype)


Accounts
opened_date : object

Transactions
txn_date : object

Loans
loan_date : object

Cards
issued_date : object


### Convert Date Columns

In [11]:
for name, columns in date_columns.items():

    for col in columns:

        cleaned_datasets[name][col] = pd.to_datetime(
            cleaned_datasets[name][col],
            format='%Y-%m-%d',
            errors='raise'
        )

    print(f"{name}: Date conversion completed")

Accounts: Date conversion completed
Transactions: Date conversion completed
Loans: Date conversion completed
Cards: Date conversion completed


### Verify Converted Datatypes

In [12]:
for name, columns in date_columns.items():

    print(f"\n{name}")

    for col in columns:
        print(col, ":", cleaned_datasets[name][col].dtype)


Accounts
opened_date : datetime64[ns]

Transactions
txn_date : datetime64[ns]

Loans
loan_date : datetime64[ns]

Cards
issued_date : datetime64[ns]


### Verify Missing Values After Conversion

In [13]:
for name, columns in date_columns.items():

    for col in columns:

        missing = cleaned_datasets[name][col].isnull().sum()

        print(f"{name} - {col}: {missing} missing values")

Accounts - opened_date: 0 missing values
Transactions - txn_date: 0 missing values
Loans - loan_date: 0 missing values
Cards - issued_date: 0 missing values


#### Date Data Type Conversion
- Converted four date columns from object to datetime64[ns].
- Used pd.to_datetime() with the expected date format.
- Verified that no missing values were introduced during conversion.
- All date columns were successfully converted.

# Numerical Data Type Validation


In [15]:
for name, df in cleaned_datasets.items():

    numeric_cols = df.select_dtypes(include=np.number).columns

    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    print(f"{'='*40}")

    for col in numeric_cols:
        print(f"{col}: {df[col].dtype}")


Dataset: Customers
customer_id: int64
birth_year: int64
county_id: int64

Dataset: Accounts
account_id: int64
county_id: int64

Dataset: Account_Holders
holder_id: int64
account_id: int64
customer_id: int64

Dataset: Transactions
transaction_id: int64
account_id: int64
amount_kes: int64
balance_kes: int64

Dataset: Loans
loan_id: int64
account_id: int64
amount_kes: int64
duration_months: int64
monthly_payment_kes: int64

Dataset: Cards
card_id: int64
holder_id: int64

Dataset: Counties
county_id: int64
population: int64
avg_monthly_income_kes: int64
unemployment_pct: float64
urban_pct: float64


No numerical datatype conversion is required.

# Object Datatype Validation

In [16]:
for name, df in cleaned_datasets.items():

    object_cols = df.select_dtypes(
        include='object'
    ).columns

    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    print(f"{'='*40}")

    for col in object_cols:
        print(f"{col}: {df[col].dtype}")


Dataset: Customers
full_name: object
gender: object

Dataset: Accounts
account_type: object
statement_frequency: object

Dataset: Account_Holders
holder_type: object

Dataset: Transactions
txn_type: object
channel: object
purpose: object

Dataset: Loans
status: object

Dataset: Cards
card_type: object

Dataset: Counties
county: object
region: object


There’s no need to convert any object column right now.

# Missing Value Handling

In [14]:
for name, df in cleaned_datasets.items():

    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    print(f"{'='*40}")

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No Missing Values Found!")
    else:
        display(missing)


Dataset: Customers
No Missing Values Found!

Dataset: Accounts
No Missing Values Found!

Dataset: Account_Holders
No Missing Values Found!

Dataset: Transactions


purpose    67264
dtype: int64


Dataset: Loans
No Missing Values Found!

Dataset: Cards
No Missing Values Found!

Dataset: Counties
No Missing Values Found!


### Missing Value Handling

The missing value analysis identified missing data only in the `purpose` column of the Transactions dataset.

- Total missing values: 67,264
- Missing percentage: 65.38%

**Cleaning Decision:**

The missing values were retained because their exact reason could not be established. Missing purposes were concentrated in specific transaction channels, and assigning assumed categories could introduce misleading information.

No missing value imputation was required in the remaining six datasets.

# Duplicate Verification After Cleaning

In [17]:
for name, df in cleaned_datasets.items():

    print(f"\n{name}")
    print("Full Row Duplicates:", df.duplicated().sum())


Customers
Full Row Duplicates: 0

Accounts
Full Row Duplicates: 0

Account_Holders
Full Row Duplicates: 0

Transactions
Full Row Duplicates: 0

Loans
Full Row Duplicates: 0

Cards
Full Row Duplicates: 0

Counties
Full Row Duplicates: 0


# Primary Key Verification

In [18]:
primary_keys = {
    'Customers': 'customer_id',
    'Accounts': 'account_id',
    'Account_Holders': 'holder_id',
    'Transactions': 'transaction_id',
    'Loans': 'loan_id',
    'Cards': 'card_id',
    'Counties': 'county_id'
}

for name, pk in primary_keys.items():

    df = cleaned_datasets[name]

    print(f"\n{name}")
    print("Primary Key Duplicates:", df[pk].duplicated().sum())
    print("Primary Key Missing:", df[pk].isnull().sum())


Customers
Primary Key Duplicates: 0
Primary Key Missing: 0

Accounts
Primary Key Duplicates: 0
Primary Key Missing: 0

Account_Holders
Primary Key Duplicates: 0
Primary Key Missing: 0

Transactions
Primary Key Duplicates: 0
Primary Key Missing: 0

Loans
Primary Key Duplicates: 0
Primary Key Missing: 0

Cards
Primary Key Duplicates: 0
Primary Key Missing: 0

Counties
Primary Key Duplicates: 0
Primary Key Missing: 0


No action required.

## Data Cleaning Decisions

Based on the Data Quality Assessment performed in Notebook 01, the following cleaning decisions were made:

| Data Quality Issue | Observation | Decision |
|---|---|---|
| Date Columns | Stored as object/string | Converted to datetime |
| Missing Values | Missing values found in Transactions[purpose] | Retained as unknown |
| Duplicate Records | No full-row duplicates found | No removal required |
| Primary Keys | No missing or duplicate primary keys | No correction required |
| Numerical Data Types | Appropriate numerical data types | No conversion required |
| Categorical Values | No obvious inconsistencies observed | No correction required |
| Outliers | High-value observations identified | Retained for analysis |

**Note:** No unnecessary modifications were made to the datasets. Only changes supported by the data quality assessment were applied.

# Save Cleaned Datasets

In [19]:
file_names = {
    'Customers': 'bank-customers.csv',
    'Accounts': 'bank-accounts.csv',
    'Account_Holders': 'bank-account-holders.csv',
    'Transactions': 'bank-transactions(1).csv',
    'Loans': 'bank-loans.csv',
    'Cards': 'bank-cards.csv',
    'Counties': 'bank-counties.csv'
}

for name, df in cleaned_datasets.items():
    df.to_csv(cleaned_path / file_names[name], index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


# Reload and Verify Cleaned Datasets

In [20]:
date_columns = {
    'Accounts': ['opened_date'],
    'Transactions': ['txn_date'],
    'Loans': ['loan_date'],
    'Cards': ['issued_date']
}

verified_datasets = {}

for name, filename in file_names.items():

    verified_datasets[name] = pd.read_csv(
        cleaned_path / filename,
        parse_dates=date_columns.get(name, [])
    )

    print(
        f"{name}: "
        f"{verified_datasets[name].shape} | "
        f"Loaded Successfully"
    )

Customers: (1034, 5) | Loaded Successfully
Accounts: (900, 5) | Loaded Successfully
Account_Holders: (1034, 4) | Loaded Successfully
Transactions: (102880, 8) | Loaded Successfully
Loans: (296, 7) | Loaded Successfully
Cards: (232, 4) | Loaded Successfully
Counties: (15, 7) | Loaded Successfully


# Verify Data Integrity

In [21]:
for name in raw_datasets:

    original = raw_datasets[name]
    cleaned = verified_datasets[name]

    print(f"\n{name}")
    print("Original Shape:", original.shape)
    print("Cleaned Shape:", cleaned.shape)
    print("Same Records:", original.equals(cleaned))


Customers
Original Shape: (1034, 5)
Cleaned Shape: (1034, 5)
Same Records: True

Accounts
Original Shape: (900, 5)
Cleaned Shape: (900, 5)
Same Records: False

Account_Holders
Original Shape: (1034, 4)
Cleaned Shape: (1034, 4)
Same Records: True

Transactions
Original Shape: (102880, 8)
Cleaned Shape: (102880, 8)
Same Records: False

Loans
Original Shape: (296, 7)
Cleaned Shape: (296, 7)
Same Records: False

Cards
Original Shape: (232, 4)
Cleaned Shape: (232, 4)
Same Records: False

Counties
Original Shape: (15, 7)
Cleaned Shape: (15, 7)
Same Records: True


In [22]:
for name in raw_datasets:

    original = raw_datasets[name].copy()
    cleaned = verified_datasets[name].copy()

    # Convert original date columns to datetime
    for col in date_columns.get(name, []):
        original[col] = pd.to_datetime(
            original[col],
            format='%Y-%m-%d'
        )

    print(f"\n{name}")
    print("Same Records:", original.equals(cleaned))


Customers
Same Records: True

Accounts
Same Records: True

Account_Holders
Same Records: True

Transactions
Same Records: True

Loans
Same Records: True

Cards
Same Records: True

Counties
Same Records: True
